# R06 — N01 Rotation–SimCLR Feature Fusion (Seed 42)

**Kaggle notebook:** `lesion-r06-n01-rotation-simclr-feature-fusion-s42`  
**Required notebook output:** completed `lesion-r05-n01-dual-ssl-featurebank-s42`. **GPU optional** (recommended); **Internet OFF**.

Four **new PyTorch feature-fusion adaptations**: FusionConcat, FusionGatedV2, FusionGatedV3, ResidualFusion. These are not claimed to reproduce the historic Keras fusion models exactly. Train fusion heads on frozen N01/R04 encoder feature vectors, selecting by validation balanced accuracy. Export meta_train predictions for later stacking; never access lesion-aware test.

## Configuration and local library checks

In [1]:
import os,json,hashlib,random,time,zipfile,shutil,math
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset,DataLoader
from sklearn.metrics import balanced_accuracy_score,accuracy_score,f1_score,precision_score,recall_score,roc_auc_score,average_precision_score,matthews_corrcoef,confusion_matrix,classification_report,log_loss
import matplotlib.pyplot as plt

NAME='lesion-r06-n01-rotation-simclr-feature-fusion-s42'
SEED=42
INPUT=Path(os.getenv('R06_INPUT_DIR','/kaggle/input' if Path('/kaggle/input').exists() else '/mnt/data/r06_test_inputs')).resolve()
OUT=Path(os.getenv('R06_OUT','/kaggle/working/'+NAME if Path('/kaggle/working').exists() else './'+NAME)).resolve()
STAGING=Path(os.getenv('R06_STAGE_DIR','/kaggle/working/.r06_input' if Path('/kaggle/working').exists() else './.r06_input')).resolve()
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC'];SOURCES=['rotation','simclr'];EXPECTED={'train':16490,'validation':2186,'meta_train':2267}
ARCHS=['FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion']
BATCH=256;MAX_EPOCHS=70;PATIENCE=12;LR=3e-4;WEIGHT_DECAY=1e-3;DROPOUT=.35
MAX_HOURS=7.7;START=time.monotonic()
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT.mkdir(exist_ok=True,parents=True)
for k in ['models','predictions','reports','tables','figures']:(OUT/k).mkdir(exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if DEVICE.type=='cuda':torch.cuda.manual_seed_all(SEED)

def req(ok,msg):
    if not bool(ok):raise RuntimeError(msg)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)
    return h.hexdigest()
def jwrite(p,obj):
    p=Path(p);p.parent.mkdir(exist_ok=True,parents=True);tmp=p.with_suffix('.tmp')
    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False,default=str));tmp.replace(p)
def checkpoint(path,obj):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_suffix('.tmp');torch.save(obj,tmp);tmp.replace(p)
print(NAME,'device:',DEVICE,'torch:',torch.__version__)


lesion-r06-n01-rotation-simclr-feature-fusion-s42 device: cuda torch: 2.11.0+cu130


## Discover the completed R05 output (extracted files, Kaggle ZIP, or nested ZIP)

In [2]:
def available(root):
    found=[]
    for p in root.rglob('final_status.json'):
        if p.parent.name!='reports':continue
        try:record=json.loads(p.read_text())
        except Exception:continue
        if record.get('stage')=='R05' and record.get('seed')==SEED:found.append(p.parent.parent)
    return sorted(set(p.resolve() for p in found))

def unpack(path,target):
    target.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(path) as z:
        infos=[x for x in z.infolist() if not x.is_dir()]
        size=sum(x.file_size for x in infos)
        req(size<min(8*2**30,int(shutil.disk_usage(target).free*.90)),'Insufficient disk for R05 feature archive')
        for f in infos:
            q=Path(f.filename.replace('\\','/'))
            req(not q.is_absolute() and '..' not in q.parts,'Unsafe ZIP member')
            dest=(target/q).resolve();req(dest.is_relative_to(target.resolve()),'Unsafe ZIP path')
            dest.parent.mkdir(exist_ok=True,parents=True)
            if dest.is_file() and dest.stat().st_size==f.file_size:continue
            with z.open(f) as a,open(dest,'wb') as b:shutil.copyfileobj(a,b,2**20)

roots=available(INPUT)
if len(roots)!=1:
    archives=sorted(INPUT.rglob('*.zip'))
    for level in range(2):
        if len(roots)==1:break
        new=[]
        for zp in archives:
            dest=STAGING/hashlib.sha256(str(zp).encode()).hexdigest()[:12]
            flag=dest/'.ok'
            if flag.exists():continue
            with zipfile.ZipFile(zp) as z:
                relevant=zp.name=='_output_.zip' or any(n.endswith('reports/final_status.json') for n in z.namelist()) or any(n.endswith('.zip') for n in z.namelist())
            if not relevant:continue
            print('Unpacking input',zp);unpack(zp,dest);flag.write_text('ok')
            new.extend(p for p in dest.rglob('*.zip') if p.is_file())
        roots=available(INPUT)+available(STAGING)
        roots=sorted(set(roots));archives=sorted(set(archives+new))
req(len(roots)==1,f'Attach only the completed R05 feature-bank output; found {roots}')
ROOT=roots[0]
status=json.loads((ROOT/'reports/final_status.json').read_text())
req(status['status']=='COMPLETED' and not status['test_touched'],'R05 incomplete or touched test')
req(status['classes']==CLASSES and status['splits']==EXPECTED and status['feature_dim_per_source']==2048,'Wrong R05 protocol')
print('R05 found:',ROOT,'R04 checkpoints:',{s:status['r04_provenance'][s]['best_epoch'] for s in SOURCES})
previous=os.getenv('R06_PREVIOUS_ROOT','').strip()
if previous:
    prev=Path(previous).expanduser().resolve()
    if prev.is_file() and zipfile.is_zipfile(prev):
        target=STAGING/'previous_r06';unpack(prev,target)
        matches=[p.parent.parent for p in target.rglob('final_status.json') if p.parent.name=='reports' and json.loads(p.read_text()).get('stage')=='R06']
        req(len(matches)==1,'R06 previous ZIP must contain exactly one R06 report')
        prev=matches[0]
    prior=json.loads((prev/'reports/final_status.json').read_text())
    req(prior.get('stage')=='R06' and prior['n01_manifest_sha256']==status['n01_manifest_sha256'],'Previous R06 dataset mismatch')
    copied=0
    for name in ARCHS:
        for f in [f'models/{name}_best.pt',f'models/{name}_last.pt',f'tables/{name}_history.csv']:
            old=prev/f;dest=OUT/f
            if old.is_file() and not dest.exists():dest.parent.mkdir(exist_ok=True,parents=True);shutil.copy2(old,dest);copied+=1
    print('R06 restored previous checkpoints/history:',copied)


R05 found: /kaggle/input/notebooks/masumbillah029/lesion-r05-n01-dual-ssl-featurebank-s42/lesion-r05-n01-dual-ssl-featurebank-s42 R04 checkpoints: {'rotation': 10, 'simclr': 14}


## Verify feature hashes, splits, and normalize using training features only

In [3]:
meta={};raw={}
for split,n in EXPECTED.items():
    p=ROOT/status['manifests'][split]['file']
    req(sha(p)==status['manifests'][split]['sha256'],f'{split} manifest corrupted')
    d=pd.read_csv(p,dtype='string',keep_default_na=False)
    req(len(d)==n and d.image_id.is_unique and d.label.isin(CLASSES).all(),f'{split} wrong manifest')
    req(d.class_index.eq(d.label.map({c:str(i) for i,c in enumerate(CLASSES)})).all(),f'{split} class mismatch')
    meta[split]=d;raw[split]={}
    for source in SOURCES:
        item=status['features'][split][source]
        p=ROOT/item['file'];req(p.is_file() and sha(p)==item['sha256'],f'{split} {source} corrupted or missing')
        z=np.load(p,mmap_mode='r')
        req(z.shape==(n,2048) and np.isfinite(z).all(),f'{split} {source} invalid shape/features')
        raw[split][source]=z
for a,b in [('train','validation'),('train','meta_train'),('validation','meta_train')]:
    for col in ['image_id','group_id','source_sha256']:
        req(set(meta[a][col]).isdisjoint(set(meta[b][col])),f'{a}/{b} overlap in {col}')
# Compute, store and apply feature channel statistics using training data only.
norm={};scaled={split:{} for split in EXPECTED}
for source in SOURCES:
    a=np.asarray(raw['train'][source],dtype=np.float64)
    mean=a.mean(0).astype('float32');sd=np.maximum(a.std(0).astype('float32'),.01)
    req(np.isfinite(mean).all() and np.isfinite(sd).all(),'Invalid normalization')
    norm[source]=(mean,sd)
    for split in EXPECTED:
        z=np.asarray(raw[split][source],dtype='float32')
        scaled[split][source]=np.clip((z-mean)/sd,-12,12).astype('float32')
    print(source,'normalized training features',scaled['train'][source].shape)
np.savez_compressed(OUT/'reports/train_only_normalization.npz',**{f'{s}_{k}':v for s,(a,b) in norm.items() for k,v in [('mean',a),('std',b)]})
y={s:meta[s].class_index.astype(int).to_numpy(dtype=np.int64) for s in EXPECTED}
counts=np.bincount(y['train'],minlength=8)
req((counts>0).all(),'Missing diagnostic class in training split')
weights=np.sqrt(len(y['train'])/counts);weights=.99*weights/weights.max()
np.save(OUT/'reports/class_weights.npy',weights.astype('float32'))
pd.DataFrame({'class':CLASSES,'count':counts,'loss_weight':weights}).to_csv(OUT/'tables/class_weights.csv',index=False)


rotation normalized training features (16490, 2048)
simclr normalized training features (16490, 2048)


## Define four distinct trainable fusion heads, no encoder retraining

In [4]:
class FusionConcat(nn.Module):
    def __init__(self):
        super().__init__();self.body=nn.Sequential(nn.LayerNorm(4096),nn.Linear(4096,512),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(512,128),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(128,8))
    def forward(self,r,s):return self.body(torch.cat([r,s],dim=1))
class FusionGatedV2(nn.Module):
    def __init__(self):
        super().__init__();self.r=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256),nn.GELU())
        self.s=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256),nn.GELU())
        self.gate=nn.Sequential(nn.Linear(512,256),nn.Sigmoid())
        self.classify=nn.Sequential(nn.LayerNorm(256),nn.Dropout(DROPOUT),nn.Linear(256,8))
    def forward(self,r,s):
        a=self.r(r);b=self.s(s);g=self.gate(torch.cat([a,b],1))
        return self.classify(g*a+(1-g)*b)
class FusionGatedV3(nn.Module):
    def __init__(self):
        super().__init__();self.r=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256),nn.GELU())
        self.s=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256),nn.GELU())
        self.gate=nn.Linear(512,512)
        self.classify=nn.Sequential(nn.LayerNorm(512),nn.Linear(512,128),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(128,8))
    def forward(self,r,s):
        a=self.r(r);b=self.s(s);g=torch.softmax(self.gate(torch.cat([a,b],1)).reshape(-1,2,256),dim=1)
        mix=g[:,0,:]*a+g[:,1,:]*b
        return self.classify(torch.cat([mix,torch.abs(a-b)],1))
class ResidualFusion(nn.Module):
    def __init__(self):
        super().__init__();self.r=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256))
        self.s=nn.Sequential(nn.LayerNorm(2048),nn.Linear(2048,256))
        self.res=nn.Sequential(nn.LayerNorm(512),nn.Linear(512,256),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(256,256))
        self.classify=nn.Sequential(nn.LayerNorm(256),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(256,8))
    def forward(self,r,s):
        a=self.r(r);b=self.s(s);return self.classify(.5*(a+b)+self.res(torch.cat([a,b],1)))
MODELS={'FusionConcat':FusionConcat,'FusionGatedV2':FusionGatedV2,'FusionGatedV3':FusionGatedV3,'ResidualFusion':ResidualFusion}
class FusedDataset(TensorDataset):pass
splits={s:TensorDataset(torch.from_numpy(scaled[s]['rotation']),torch.from_numpy(scaled[s]['simclr']),torch.from_numpy(y[s])) for s in EXPECTED}
loader_val=DataLoader(splits['validation'],batch_size=512,shuffle=False)
loader_meta=DataLoader(splits['meta_train'],batch_size=512,shuffle=False)
class_w=torch.from_numpy(weights.astype('float32')).to(DEVICE)
@torch.inference_mode()
def predict(model,loader):
    model.eval();yz=[];logits=[]
    for r,s,yy in loader:
        z=model(r.to(DEVICE),s.to(DEVICE)).float()
        req(torch.isfinite(z).all(),'Nonfinite fusion logits')
        yz.append(yy.numpy());logits.append(z.cpu().numpy())
    yy=np.concatenate(yz);z=np.concatenate(logits)
    pr=torch.softmax(torch.tensor(z),dim=1).numpy()
    return yy,z,pr

def score(yy,pr):
    pred=pr.argmax(axis=1)
    return {'accuracy':float(accuracy_score(yy,pred)),
        'balanced_accuracy':float(balanced_accuracy_score(yy,pred)),
        'macro_f1':float(f1_score(yy,pred,average='macro',labels=np.arange(8),zero_division=0)),
        'macro_precision':float(precision_score(yy,pred,average='macro',labels=np.arange(8),zero_division=0)),
        'macro_sensitivity':float(recall_score(yy,pred,average='macro',labels=np.arange(8),zero_division=0)),
        'macro_specificity':float(np.mean([((yy!=j)&(pred!=j)).sum()/max(1,(yy!=j).sum()) for j in range(8)])),
        'macro_auc':float(roc_auc_score(yy,pr,multi_class='ovr',average='macro',labels=np.arange(8))),
        'macro_pr_auc':float(np.mean([average_precision_score((yy==k).astype(int),pr[:,k]) for k in range(8)])),
        'mcc':float(matthews_corrcoef(yy,pred)),
        'nll':float(log_loss(yy,pr,labels=list(range(8))))}
for name,C in MODELS.items():
    m=C().to(DEVICE);r=torch.randn(3,2048,device=DEVICE);s=torch.randn(3,2048,device=DEVICE)
    z=m(r,s);req(z.shape==(3,8) and z.isfinite().all(),f'{name} shape/finite failure')
    loss=(F.cross_entropy(z,torch.tensor([0,1,2],device=DEVICE),reduction='none')*class_w[[0,1,2]]).mean();loss.backward()
    req(all(p.grad is None or p.grad.isfinite().all() for p in m.parameters()),f'{name} gradient failed')
    print('PREFLIGHT',name,'params',sum(p.numel() for p in m.parameters()))


PREFLIGHT FusionConcat params 2172552
PREFLIGHT FusionGatedV2 params 1191176
PREFLIGHT FusionGatedV3 params 1387656
PREFLIGHT ResidualFusion params 1257992


## Train, checkpoint and validate each fusion head; meta_train inference only

In [5]:
summary=[]
for i,name in enumerate(ARCHS):
    seed=SEED+i*101;random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if DEVICE.type=='cuda':torch.cuda.manual_seed_all(seed)
    model=MODELS[name]().to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=.5,patience=4,min_lr=1e-6)
    bpath=OUT/'models'/f'{name}_best.pt';lpath=OUT/'models'/f'{name}_last.pt';hpath=OUT/'tables'/f'{name}_history.csv'
    config={'name':name,'seed':seed,'lr':LR,'dropout':DROPOUT,'max_epochs':MAX_EPOCHS,'patience':PATIENCE,
            'r05_manifest_sha256':status['n01_manifest_sha256'],'r05_provenance':status['r04_provenance'],
            'r05_feature_sha256':{sp:{so:status['features'][sp][so]['sha256'] for so in SOURCES} for sp in EXPECTED},
            'classes':CLASSES,'fit_partition':'train','selection_partition':'validation','meta_fit':False,'test_access':False}
    sig=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
    best_key=(-1.,-1.,-1.);bad=0;start_ep=0;history=[]
    if lpath.exists():
        saved=torch.load(lpath,map_location='cpu',weights_only=False)
        req(saved['signature']==sig,f'{name} resume config/provenance mismatch')
        model.load_state_dict(saved['model']);opt.load_state_dict(saved['optimizer']);scheduler.load_state_dict(saved['scheduler'])
        best_key=tuple(saved['best_key']);bad=saved['bad'];start_ep=saved['next_epoch'];history=saved['history']
        req(bpath.exists() if best_key[0]>=0 else True,f'{name} lost best checkpoint')
        print('RESUME',name,'from epoch',start_ep+1)
    for epoch in range(start_ep,MAX_EPOCHS):
        if time.monotonic()-START>MAX_HOURS*3600:
            print('RUNTIME GUARD',name,'next_epoch',epoch+1);break
        if bad>=PATIENCE:
            print('EARLY STOP',name);break
        model.train();gen=torch.Generator().manual_seed(seed*10000+epoch)
        loader=DataLoader(splits['train'],batch_size=BATCH,shuffle=True,generator=gen)
        total=0.;n=0;t0=time.monotonic()
        for r,s,yy in loader:
            r=r.to(DEVICE);s=s.to(DEVICE);yy=yy.to(DEVICE)
            opt.zero_grad(set_to_none=True);z=model(r,s)
            loss=(F.cross_entropy(z,yy,reduction='none')*class_w[yy]).mean()
            req(torch.isfinite(loss),'Nonfinite fusion training loss')
            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            req(all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters()),'Nonfinite gradient')
            opt.step();total+=loss.item()*len(yy);n+=len(yy)
        req(n==len(splits['train']),'Incomplete epoch')
        yy,zz,pp=predict(model,loader_val);metrics=score(yy,pp)
        scheduler.step(metrics['nll']);key=(metrics['balanced_accuracy'],metrics['macro_f1'],metrics['accuracy'])
        improved=key>best_key
        if improved:
            best_key=key;bad=0
            checkpoint(bpath,{'model':{k:v.cpu().detach().clone() for k,v in model.state_dict().items()},
               'signature':sig,'config':config,'best_epoch':epoch+1,'metrics':metrics})
        else:bad+=1
        history.append({'epoch':epoch+1,'train_loss':total/n,'val_bacc':metrics['balanced_accuracy'],'val_macro_f1':metrics['macro_f1'],
                        'val_accuracy':metrics['accuracy'],'val_nll':metrics['nll'],'lr':opt.param_groups[0]['lr'],
                        'best':int(improved),'elapsed_sec':round(time.monotonic()-t0,2)})
        checkpoint(lpath,{'signature':sig,'next_epoch':epoch+1,'best_key':list(best_key),'bad':bad,'history':history,
                          'model':{k:v.cpu().detach().clone() for k,v in model.state_dict().items()},
                          'optimizer':opt.state_dict(),'scheduler':scheduler.state_dict()})
        pd.DataFrame(history).to_csv(hpath,index=False)
        print(name,epoch+1,'BACC',round(metrics['balanced_accuracy']*100,2),'F1',round(metrics['macro_f1']*100,2),'*' if improved else '')
    finished=(bad>=PATIENCE or len(history)>=MAX_EPOCHS)
    if not finished:
        summary.append({'model':name,'status':'PAUSED','best_bacc':best_key[0]});print('R06 PAUSED after',len(history),'epochs for',name);break
    if not bpath.exists():
        summary.append({'model':name,'status':'PAUSED','best_bacc':None});break
    best=torch.load(bpath,map_location='cpu',weights_only=False)
    req(best['signature']==sig,'Wrong selected best checkpoint')
    model.load_state_dict(best['model']);model.eval()
    final={}
    for split,loader in [('validation',loader_val),('meta_train',loader_meta)]:
        yy,z,pr=predict(model,loader)
        req(np.array_equal(yy,y[split]),f'{name} {split}: incorrect image order')
        np.savez_compressed(OUT/'predictions'/f'{name}_{split}.npz',image_id=meta[split].image_id.to_numpy(dtype=str),
            group_id=meta[split].group_id.to_numpy(dtype=str),y=yy.astype(np.int32),logits=z.astype('float32'),
            proba=pr.astype('float32'),classes=np.array(CLASSES),model_id=np.array(name),seed=np.array(seed),
            train_partition=np.array('train'),selection_partition=np.array('validation'))
        final[split]=score(yy,pr)
    cm=confusion_matrix(y['validation'],np.load(OUT/'predictions'/f'{name}_validation.npz')['proba'].argmax(1),labels=np.arange(8))
    pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(OUT/'tables'/f'{name}_confusion_matrix.csv')
    pd.DataFrame(classification_report(y['validation'],np.load(OUT/'predictions'/f'{name}_validation.npz')['proba'].argmax(1),labels=np.arange(8),target_names=CLASSES,output_dict=True,zero_division=0)).T.to_csv(OUT/'tables'/f'{name}_class_report.csv')
    jwrite(OUT/'reports'/f'{name}_metrics.json',{'stage':'R06','name':name,'seed':seed,'status':'COMPLETED',
        'best_epoch':best['best_epoch'],'selected_validation':final['validation'],'meta_train_inference_only':True,
        'meta_train_diagnostic':final['meta_train'],'configuration':config})
    head_parameters=sum(p.numel() for p in model.parameters())
    summary.append({'model':name,'status':'COMPLETED','best_epoch':best['best_epoch'],
                    'fusion_head_params_M':round(head_parameters/1e6,3),
                    'full_inference_params_M':round((2*23508032+head_parameters)/1e6,3),
                    **{k:round(v,6) for k,v in final['validation'].items()}})
    del model;torch.cuda.empty_cache() if DEVICE.type=='cuda' else None
report=pd.DataFrame(summary);report.to_csv(OUT/'tables/fusion_validation_comparison.csv',index=False)
print(report.to_string(index=False))
completed=all(x['status']=='COMPLETED' for x in summary)
jwrite(OUT/'reports/final_status.json',{'stage':'R06','seed':SEED,'status':'COMPLETED' if completed else 'PAUSED',
    'n01_manifest_sha256':status['n01_manifest_sha256'],'r05_source':str(ROOT),
    'r04_encoder_provenance':status['r04_provenance'],'test_touched':False,
    'note':'Fusion trained on R00 train; validation selects checkpoints and models; meta_train inference only for subsequent leak-safe ensemble work.'})
print('R06 finished; status', 'COMPLETED' if completed else 'PAUSED','output',OUT)


FusionConcat 1 BACC 58.14 F1 54.12 *
FusionConcat 2 BACC 57.97 F1 54.22 
FusionConcat 3 BACC 58.09 F1 54.53 
FusionConcat 4 BACC 56.85 F1 53.23 
FusionConcat 5 BACC 57.7 F1 53.79 
FusionConcat 6 BACC 57.11 F1 54.88 
FusionConcat 7 BACC 56.17 F1 54.38 
FusionConcat 8 BACC 56.47 F1 54.97 
FusionConcat 9 BACC 54.83 F1 54.05 
FusionConcat 10 BACC 55.74 F1 54.37 
FusionConcat 11 BACC 55.58 F1 53.56 
FusionConcat 12 BACC 55.77 F1 53.96 
FusionConcat 13 BACC 55.95 F1 54.65 
EARLY STOP FusionConcat
FusionGatedV2 1 BACC 57.14 F1 54.51 *
FusionGatedV2 2 BACC 56.66 F1 54.35 
FusionGatedV2 3 BACC 58.18 F1 54.93 *
FusionGatedV2 4 BACC 57.92 F1 54.87 
FusionGatedV2 5 BACC 56.39 F1 54.07 
FusionGatedV2 6 BACC 58.36 F1 55.35 *
FusionGatedV2 7 BACC 55.11 F1 54.19 
FusionGatedV2 8 BACC 56.19 F1 54.26 
FusionGatedV2 9 BACC 55.97 F1 54.51 
FusionGatedV2 10 BACC 56.29 F1 55.41 
FusionGatedV2 11 BACC 55.97 F1 53.98 
FusionGatedV2 12 BACC 56.0 F1 54.74 
FusionGatedV2 13 BACC 55.37 F1 54.3 
FusionGatedV2 14 B